# 02 — Data preparation and grouped splitting

The CSVs are already synchronized/interpolated and track-relative source files already exist. For model CV this workflow reloads raw telemetry and recalibrates the coordinate transform from the training lap only, then applies that transform to the validation lap. Lap identity is a split group, never a predictor. A median imputer and optional scaler are fitted inside each estimator pipeline.

In [ ]:
from ml_analysis.preprocessing import load_raw_laps, group_splits, fold_dataset
import numpy as np

laps = load_raw_laps((5, 11))
lap_ids = np.asarray(sorted(laps))
splits = group_splits(lap_ids, n_splits=2)
folds = []
for train_idx, validation_idx in splits:
    train_laps = tuple(int(v) for v in lap_ids[train_idx])
    validation_laps = tuple(int(v) for v in lap_ids[validation_idx])
    folds.append(fold_dataset(laps, train_laps, validation_laps))
[(f['transform_fitted_laps'], np.unique(f['groups_validation']).tolist(), f['X_train'].shape, f['X_validation'].shape) for f in folds]

Only two independent laps are available for model development, so two-fold GroupKFold is the maximum meaningful N-fold split here. The transforms avoid the historical shared-lap calibration leakage, but the tiny group count still yields high uncertainty. Missing saved values are zero; source metadata explains interpolation across nonuniform native streams. We preserve possible physical outliers rather than drop them without evidence.